In [9]:
import pandas as pd

df=pd.read_csv("../Data/Churn_Data.csv")
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [10]:
print(df.columns.tolist())

['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [12]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [13]:
pd.to_numeric(df['TotalCharges'],errors='coerce').isnull().sum()

np.int64(11)

In [14]:
df[pd.to_numeric(df['TotalCharges'], errors='coerce').isnull()][['tenure', 'MonthlyCharges', 'TotalCharges']]

,tenure,MonthlyCharges,TotalCharges
488,0,52.55,
753,0,20.25,
936,0,80.85,
1082,0,25.75,
1340,0,56.05,
3331,0,19.85,
3826,0,25.35,
4380,0,20.00,
5218,0,19.70,
6670,0,73.35,


In [15]:
df=df.drop(columns='TotalCharges')
print(df.shape)

(7043, 20)


In [16]:
df=df.rename(columns={'MonthlyCharges':'transaction_amount',
'tenure':'customer_tenure_months'})

In [17]:
df['transaction_amount'] = df['transaction_amount'] * 12

In [18]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'customer_tenure_months', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'transaction_amount', 'Churn'],
      dtype='object')

In [19]:
#keeping only the columns that are relevant to the model
df_cleaned=df[['customerID','transaction_amount','customer_tenure_months','PaymentMethod']].copy()


In [20]:
df_cleaned=df_cleaned.rename(columns={'PaymentMethod':'payment_method'})

In [21]:
print(df_cleaned.shape)
df_cleaned.head()

(7043, 4)


,customerID,transaction_amount,customer_tenure_months,payment_method
0,7590-VHVEG,358.2,1,Electronic check
1,5575-GNVDE,683.4,34,Mailed check
2,3668-QPYBK,646.2,2,Mailed check
3,7795-CFOCW,507.6,45,Bank transfer (automatic)
4,9237-HQITU,848.4,2,Electronic check


In [22]:
import numpy as np

# Set a seed so your random results are reproducible - 
# meaning if you run this again, you get the SAME "random" numbers.
# This matters for reproducibility when you write your README/report later.
np.random.seed(42)

# We want more 0s and 1s, fewer 2s and 3s - reflecting real-world retry patterns
retry_options = [0, 1, 2, 3]
retry_probabilities = [0.45, 0.30, 0.15, 0.10]  # must sum to 1.0

df_cleaned['retry_count'] = np.random.choice(
    retry_options, 
    size=len(df_cleaned), 
    p=retry_probabilities
)

print(df_cleaned['retry_count'].value_counts(normalize=True))

retry_count
0    0.456766
1    0.294903
2    0.149652
3    0.098680
Name: proportion, dtype: float64


In [23]:
def assign_failure_reason(amount):
    # Higher-amount transactions are more likely to fail due to insufficient funds
    if amount > 70:
        probabilities = {
            'insufficient_funds': 0.45,
            'card_expired': 0.20,
            'network_error': 0.15,
            'bank_decline': 0.20
        }
    else:
        # Lower-amount transactions: failure reasons are more evenly spread
        probabilities = {
            'insufficient_funds': 0.25,
            'card_expired': 0.25,
            'network_error': 0.25,
            'bank_decline': 0.25
        }
    
    reasons = list(probabilities.keys())
    probs = list(probabilities.values())
    return np.random.choice(reasons, p=probs)

# Apply this function row-by-row, based on each row's own transaction_amount
df_cleaned['failure_reason'] = df_cleaned['transaction_amount'].apply(assign_failure_reason)

print(df_cleaned['failure_reason'].value_counts(normalize=True))
print()
print(df_cleaned.groupby('failure_reason')['transaction_amount'].mean())

failure_reason
insufficient_funds    0.458043
card_expired          0.200341
bank_decline          0.197643
network_error         0.143973
Name: proportion, dtype: float64

failure_reason
bank_decline          769.333621
card_expired          784.685613
insufficient_funds    775.237198
network_error         783.412426
Name: transaction_amount, dtype: float64


In [24]:
df_cleaned.columns
df_cleaned.head()

,customerID,transaction_amount,customer_tenure_months,payment_method,retry_count,failure_reason
0,7590-VHVEG,358.2,1,Electronic check,0,insufficient_funds
1,5575-GNVDE,683.4,34,Mailed check,3,bank_decline
2,3668-QPYBK,646.2,2,Mailed check,1,insufficient_funds
3,7795-CFOCW,507.6,45,Bank transfer (automatic),1,network_error
4,9237-HQITU,848.4,2,Electronic check,0,insufficient_funds


In [25]:
def assign_response_time(failure_reason):
    if failure_reason == 'network_error':
        # Network errors take longer - simulate with a higher mean and more spread
        return int(np.random.normal(loc=2500, scale=800))
    else:
        # Other failures are quick, clean rejections
        return int(np.random.normal(loc=400, scale=150))

df_cleaned['gateway_response_time_ms'] = df_cleaned['failure_reason'].apply(assign_response_time)

# Response times can't be negative - clip any accidental negative values from the normal distribution to a small positive floor
df_cleaned['gateway_response_time_ms'] = df_cleaned['gateway_response_time_ms'].clip(lower=50)

print(df_cleaned.groupby('failure_reason')['gateway_response_time_ms'].mean())

failure_reason
bank_decline           400.580460
card_expired           405.023388
insufficient_funds     402.979231
network_error         2538.129191
Name: gateway_response_time_ms, dtype: float64


In [26]:
def assign_customer_segment(row):
    tenure = row['customer_tenure_months']
    amount = row['transaction_amount']
    
    if tenure <= 3:
        return 'new_customer'
    elif tenure > 24 and amount > 60:
        return 'high_value_repeat'
    else:
        return 'occasional'

df_cleaned['customer_segment'] = df_cleaned.apply(assign_customer_segment, axis=1)

print(df_cleaned['customer_segment'].value_counts(normalize=True))

customer_segment
high_value_repeat    0.544228
occasional           0.304984
new_customer         0.150788
Name: proportion, dtype: float64


In [27]:
df_cleaned.columns
df_cleaned.head(10)

,customerID,transaction_amount,customer_tenure_months,payment_method,retry_count,failure_reason,gateway_response_time_ms,customer_segment
0,7590-VHVEG,358.2,1,Electronic check,0,insufficient_funds,272,new_customer
1,5575-GNVDE,683.4,34,Mailed check,3,bank_decline,544,high_value_repeat
2,3668-QPYBK,646.2,2,Mailed check,1,insufficient_funds,404,new_customer
3,7795-CFOCW,507.6,45,Bank transfer (automatic),1,network_error,2193,high_value_repeat
4,9237-HQITU,848.4,2,Electronic check,0,insufficient_funds,382,new_customer
5,9305-CDSKC,1195.8,8,Electronic check,0,insufficient_funds,473,occasional
6,1452-KIOVK,1069.2,22,Credit card (automatic),0,card_expired,466,occasional
7,6713-OKOMC,357.0,10,Mailed check,2,bank_decline,388,occasional
8,7892-POOKP,1257.6,28,Electronic check,1,card_expired,563,high_value_repeat
9,6388-TABGU,673.8,62,Bank transfer (automatic),1,bank_decline,545,high_value_repeat


In [28]:
def assign_recovery_probability(row):
    reason = row['failure_reason']
    retries = row['retry_count']
    segment = row['customer_segment']
    
    if reason == 'card_expired':
        base_prob = 0.05  # almost never recovers without card update
    elif reason == 'network_error':
        base_prob = 0.80  # usually a transient glitch
    elif reason == 'insufficient_funds':
        # More retries already used = lower remaining chance
        base_prob = 0.65 - (retries * 0.15)
    else:  # bank_decline
        base_prob = 0.50
    
    # Small boost for high-value repeat customers (more reliable payers historically)
    if segment == 'high_value_repeat':
        base_prob += 0.05
    
    # Keep probability within valid bounds [0, 1]
    base_prob = min(max(base_prob, 0.02), 0.95)
    
    # Use this probability to randomly decide Yes/No for THIS specific row
    return np.random.choice(['Yes', 'No'], p=[base_prob, 1 - base_prob])

df_cleaned['payment_recovered'] = df_cleaned.apply(assign_recovery_probability, axis=1)

print(df_cleaned['payment_recovered'].value_counts(normalize=True))
print()
print(df_cleaned.groupby('failure_reason')['payment_recovered'].apply(lambda x: (x == 'Yes').mean()))

payment_recovered
No     0.504615
Yes    0.495385
Name: proportion, dtype: float64

failure_reason
bank_decline          0.534483
card_expired          0.071580
insufficient_funds    0.557037
network_error         0.835306
Name: payment_recovered, dtype: float64


In [29]:
print(df_cleaned.shape)
print(df_cleaned.dtypes)
df_cleaned.head()

(7043, 9)
customerID                   object
transaction_amount          float64
customer_tenure_months        int64
payment_method               object
retry_count                   int64
failure_reason               object
gateway_response_time_ms      int64
customer_segment             object
payment_recovered            object
dtype: object


,customerID,transaction_amount,customer_tenure_months,payment_method,retry_count,failure_reason,gateway_response_time_ms,customer_segment,payment_recovered
0,7590-VHVEG,358.2,1,Electronic check,0,insufficient_funds,272,new_customer,Yes
1,5575-GNVDE,683.4,34,Mailed check,3,bank_decline,544,high_value_repeat,Yes
2,3668-QPYBK,646.2,2,Mailed check,1,insufficient_funds,404,new_customer,No
3,7795-CFOCW,507.6,45,Bank transfer (automatic),1,network_error,2193,high_value_repeat,Yes
4,9237-HQITU,848.4,2,Electronic check,0,insufficient_funds,382,new_customer,No


In [30]:
df_cleaned.to_csv('../Data/payments_data_prepared.csv', index=False)
print("Saved successfully")

Saved successfully


In [31]:
df_cleaned['transaction_amount'].describe()

count    7043.000000
mean      777.140310
std       361.080565
min       219.000000
25%       426.000000
50%       844.200000
75%      1078.200000
max      1425.000000
Name: transaction_amount, dtype: float64